# Silver Layer — Clean and Transform Transactions

Transform raw banking transaction events from the Bronze Delta layer into structured, validated records.

**Processing steps**
- Parse the raw transaction JSON using an explicit schema.
- Convert the amount and timestamp into appropriate data types.
- Remove records missing required fields.
- Deduplicate transactions using `transaction_id` with a 10-minute event-time watermark.
- Write clean records to the Silver Delta layer in ADLS Gen2.

**Input:** Bronze Delta data  
**Output:** Silver Delta data  
**Storage:** Azure Data Lake Storage Gen2

## Silver Layer — Storage Access Validation

Verify that the Silver container is accessible before starting the transformation stream.

In [0]:
silver_path = (
    "abfss://silver@stbankingfrauddetection.dfs.core.windows.net/"
    "banking_transactions/"
)

display(
    dbutils.fs.ls(
        "abfss://silver@stbankingfrauddetection.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://silver@stbankingfrauddetection.dfs.core.windows.net/banking_transactions/,banking_transactions/,0,1792142985000
abfss://silver@stbankingfrauddetection.dfs.core.windows.net/checkpoints/,checkpoints/,0,1792142955000


### Read and transform Bronze data

In [0]:
from pyspark.sql.functions import (
    col, from_json, to_timestamp, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, BooleanType
)

# Bronze and Silver paths
bronze_path = (
    "abfss://bronze@stbankingfrauddetection.dfs.core.windows.net/"
    "banking_transactions/"
)

silver_path = (
    "abfss://silver@stbankingfrauddetection.dfs.core.windows.net/"
    "banking_transactions/"
)

silver_checkpoint = (
    "abfss://silver@stbankingfrauddetection.dfs.core.windows.net/"
    "checkpoints/silver_transformation_v1/"
)

# Define the expected transaction JSON schema
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("merchant", StringType(), True),
    StructField("location", StringType(), True),
    StructField("transaction_timestamp", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("is_fraud", BooleanType(), True),
    StructField("transaction_type", StringType(), True)
])

# Read the Bronze Delta stream
bronze_stream = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

# Parse raw JSON into structured columns
parsed_stream = (
    bronze_stream
    .withColumn(
        "transaction",
        from_json(col("transaction_json"), transaction_schema)
    )
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        "ingestion_timestamp",
        col("transaction.transaction_id").alias("transaction_id"),
        col("transaction.account_id").alias("account_id"),
        col("transaction.amount").alias("amount"),
        col("transaction.merchant").alias("merchant"),
        col("transaction.location").alias("location"),
       to_timestamp(
        col("transaction.transaction_timestamp")).alias("transaction_timestamp"),
        col("transaction.device_id").alias("device_id"),
        col("transaction.is_fraud").alias("is_fraud"),
        col("transaction.transaction_type").alias("transaction_type")
    )
)

# Keep records with required fields and valid amount
clean_stream = (
    parsed_stream
    .filter(
        col("transaction_id").isNotNull()
        & col("account_id").isNotNull()
        & col("amount").isNotNull()
        & (col("amount") > 0)
        & col("transaction_timestamp").isNotNull()
    )
    .withWatermark("transaction_timestamp", "10 minutes")
    .dropDuplicates(["transaction_id"])
    .withColumn("silver_processed_at", current_timestamp())
)

print("Silver transformation is configured.")

Silver transformation is configured.


### Write the Silver stream

In [0]:
silver_query = (
    clean_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("path", silver_path)
    .option("checkpointLocation", silver_checkpoint)
    .start()
)

print("Silver transformation stream started.")

Silver transformation stream started.


### Expected output

In [0]:
display(
    spark.read.format("delta")
    .load(silver_path)
    .limit(10)
)

topic,partition,offset,kafka_timestamp,ingestion_timestamp,transaction_id,account_id,amount,merchant,location,transaction_timestamp,device_id,is_fraud,transaction_type,silver_processed_at
banking-transactions,1,655,2026-10-19T07:50:57.112Z,2026-10-19T07:51:07.34Z,857e75fe-4714-4b82-bdd4-1c620d7fe029,ACC1008,2642.34,Electronics Store,Chennai,2026-10-19T07:50:56.721231Z,DEV111,false,NORMAL,2026-10-19T07:51:37.573Z
banking-transactions,1,607,2026-10-19T07:47:23.176Z,2026-10-19T07:47:24.263Z,43c7f637-72f2-45da-9a75-9a7945139186,ACC1018,4380.5,Electronics Store,Chennai,2026-10-19T07:47:23.031941Z,DEV108,false,NORMAL,2026-10-19T07:47:58.198Z
banking-transactions,0,564,2026-10-19T07:44:10.141Z,2026-10-19T07:44:10.418Z,232e74c6-cc26-4f9d-bd48-9b71db9de052,ACC1017,9942.57,Online Shopping,Chennai,2026-10-19T07:44:09.996347Z,DEV111,false,NORMAL,2026-10-19T07:45:01.448Z
banking-transactions,0,597,2026-10-19T07:46:36.971Z,2026-10-19T07:46:39.619Z,00c6fbba-6f42-4a72-8cac-b83dc2b90695,ACC1003,1243.32,Online Shopping,Chennai,2026-10-19T07:46:36.80947Z,DEV120,false,NORMAL,2026-10-19T07:46:45.813Z
banking-transactions,0,514,2026-10-19T07:40:34.465Z,2026-10-19T07:40:37.43Z,c7151b88-c2aa-46e8-8148-f58edb71eaec,ACC1010,291.51,Grocery Store,Chennai,2026-10-19T07:40:34.34111Z,DEV109,false,NORMAL,2026-10-19T07:41:04.887Z
banking-transactions,0,618,2026-10-19T07:48:11.006Z,2026-10-19T07:48:13.234Z,f5216848-c173-4af3-8c6d-8c6c0fec1a40,ACC1011,9982.55,Fuel Station,Chennai,2026-10-19T07:48:10.863279Z,DEV105,false,NORMAL,2026-10-19T07:49:05.026Z
banking-transactions,0,577,2026-10-19T07:45:09.636Z,2026-10-19T07:45:10.395Z,b072b8ad-3b52-47ad-bbd6-5634b296b517,ACC1020,763.32,Fuel Station,Chennai,2026-10-19T07:45:09.507535Z,DEV120,false,NORMAL,2026-10-19T07:45:41.122Z
banking-transactions,1,629,2026-10-19T07:49:01.9Z,2026-10-19T07:49:05.152Z,734c9f8b-8539-4c85-a3be-1df30ffb6c20,ACC1012,7079.93,Fuel Station,Chennai,2026-10-19T07:49:01.462509Z,DEV111,false,NORMAL,2026-10-19T07:49:44.037Z
banking-transactions,1,601,2026-10-19T07:46:57.218Z,2026-10-19T07:46:59.412Z,99ccca0d-e19a-4d4a-83c0-eba21e8873c4,ACC1004,721.29,Restaurant,Chennai,2026-10-19T07:46:57.0244Z,DEV117,false,NORMAL,2026-10-19T07:47:20.591Z
banking-transactions,1,654,2026-10-19T07:50:52.627Z,2026-10-19T07:51:07.34Z,ec77c63e-9260-48bb-bc19-bc9eb2f47e5a,ACC1019,2940.75,Restaurant,Chennai,2026-10-19T07:50:51.936349Z,DEV112,false,NORMAL,2026-10-19T07:51:37.573Z


## Silver Layer — Validate Bronze JSON Schema

Inspect raw transaction events from Bronze and verify that the transaction JSON matches the expected Silver schema before filtering records.

In [0]:
from pyspark.sql.functions import col, from_json

# Read a small batch from Bronze
bronze_batch = (
    spark.read
    .format("delta")
    .load(bronze_path)
)

# Inspect raw JSON records
display(
    bronze_batch
    .select("transaction_json")
    .limit(5)
)

# Test parsing against the existing transaction schema
parsed_test = bronze_batch.select(
    col("transaction_json"),
    from_json(
        col("transaction_json"),
        transaction_schema
    ).alias("parsed")
)

display(
    parsed_test
    .select(
        "transaction_json",
        "parsed.*"
    )
    .limit(5)
)

transaction_json
"{""transaction_id"": ""97d687cb-0b98-4656-b69a-1c71c3871c03"", ""account_id"": ""ACC1007"", ""amount"": 3408.78, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:58:36.829111+00:00"", ""device_id"": ""DEV101"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""4d72e3d5-6035-404f-a36c-41115ba008b9"", ""account_id"": ""ACC1002"", ""amount"": 3673.48, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:12:54.249799+00:00"", ""device_id"": ""DEV106"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""dfdd6342-25db-444f-a8a3-522a84a79c67"", ""account_id"": ""ACC1012"", ""amount"": 5060.0, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:08:58.307081+00:00"", ""device_id"": ""DEV101"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"
"{""transaction_id"": ""ab5e30cd-359d-4072-a2ef-484b5e5e2f2f"", ""account_id"": ""ACC1002"", ""amount"": 80336.15, ""merchant"": ""Restaurant"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:57:35.342431+00:00"", ""device_id"": ""DEV113"", ""is_fraud"": true, ""transaction_type"": ""HIGH_VALUE""}"
"{""transaction_id"": ""03c5db21-5de4-4a10-a7a1-7d88c123b4d7"", ""account_id"": ""ACC1008"", ""amount"": 8955.69, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:39:30.959625+00:00"", ""device_id"": ""DEV112"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"


transaction_json,transaction_id,account_id,amount,merchant,location,transaction_timestamp,device_id,is_fraud,transaction_type
"{""transaction_id"": ""1d47a3dc-cfc7-41f1-a35e-2d806b6a6825"", ""account_id"": ""ACC1001"", ""amount"": 7020.37, ""merchant"": ""Electronics Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:05:32.821213+00:00"", ""device_id"": ""DEV115"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",1d47a3dc-cfc7-41f1-a35e-2d806b6a6825,ACC1001,7020.37,Electronics Store,Chennai,2026-10-20T14:05:32.821213+00:00,DEV115,false,NORMAL
"{""transaction_id"": ""5920550b-9bf7-4bde-8d2e-b2727d4af0d9"", ""account_id"": ""ACC1004"", ""amount"": 4593.51, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:05:38.797269+00:00"", ""device_id"": ""DEV116"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",5920550b-9bf7-4bde-8d2e-b2727d4af0d9,ACC1004,4593.51,Online Shopping,Chennai,2026-10-20T14:05:38.797269+00:00,DEV116,false,NORMAL
"{""transaction_id"": ""1afda11f-bd9c-4b75-ab5a-5efe47cbb089"", ""account_id"": ""ACC1007"", ""amount"": 1197.39, ""merchant"": ""Restaurant"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:05:42.981773+00:00"", ""device_id"": ""DEV101"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",1afda11f-bd9c-4b75-ab5a-5efe47cbb089,ACC1007,1197.39,Restaurant,Chennai,2026-10-20T14:05:42.981773+00:00,DEV101,false,NORMAL
"{""transaction_id"": ""799b3ede-47ba-4547-874e-99b9e4485a4f"", ""account_id"": ""ACC1019"", ""amount"": 6223.74, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:05:47.140379+00:00"", ""device_id"": ""DEV110"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",799b3ede-47ba-4547-874e-99b9e4485a4f,ACC1019,6223.74,Online Shopping,Chennai,2026-10-20T14:05:47.140379+00:00,DEV110,false,NORMAL
"{""transaction_id"": ""35a9d20f-e792-49cf-b87c-710bcb6e8c91"", ""account_id"": ""ACC1014"", ""amount"": 246.09, ""merchant"": ""Grocery Store"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-20T14:05:51.301074+00:00"", ""device_id"": ""DEV100"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}",35a9d20f-e792-49cf-b87c-710bcb6e8c91,ACC1014,246.09,Grocery Store,Chennai,2026-10-20T14:05:51.301074+00:00,DEV100,false,NORMAL


## Diagnose Transaction Timestamp

Inspect a raw transaction event to identify the timestamp field name and format before updating the Silver schema.

In [0]:
display(
    bronze_batch
    .select("transaction_json")
    .limit(1)
)

transaction_json
"{""transaction_id"": ""4d72e3d5-6035-404f-a36c-41115ba008b9"", ""account_id"": ""ACC1002"", ""amount"": 3673.48, ""merchant"": ""Online Shopping"", ""location"": ""Chennai"", ""transaction_timestamp"": ""2026-10-19T07:12:54.249799+00:00"", ""device_id"": ""DEV106"", ""is_fraud"": false, ""transaction_type"": ""NORMAL""}"


# Gold Layer — Prepare Fraud Analytics

Inspect the Silver Delta table to identify the available transaction fields and data types before creating business-level aggregations.

**Input:** Silver Delta Lake
**Output:** Schema and sample records for Gold design

### Check schema

In [0]:
silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

silver_df.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- merchant: string (nullable = true)
 |-- location: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- device_id: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



### View sample records

In [0]:
display(silver_df.limit(10))

topic,partition,offset,kafka_timestamp,ingestion_timestamp,transaction_id,account_id,amount,merchant,location,transaction_timestamp,device_id,is_fraud,transaction_type,silver_processed_at
banking-transactions,0,470,2026-10-19T07:15:39.895Z,2026-10-19T07:15:42.985Z,e17ba7d1-4dac-4698-b7f8-755b7683dffa,ACC1003,4634.79,Restaurant,Hyderabad,2026-10-19T07:15:39.799342Z,DEV120,true,UNUSUAL_LOCATION,2026-10-19T07:16:03.817Z
banking-transactions,1,472,2026-10-19T07:15:46.222Z,2026-10-19T07:15:46.444Z,4630814e-271d-4823-8d4e-ee6a327be06d,ACC1016,9299.54,Electronics Store,Chennai,2026-10-19T07:15:46.103761Z,DEV104,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,0,473,2026-10-19T07:15:53.259Z,2026-10-19T07:15:53.455Z,727a5bd0-6ab9-44c3-94a6-d436b4fc138a,ACC1013,6456.76,Online Shopping,Chennai,2026-10-19T07:15:53.13058Z,DEV111,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,1,469,2026-10-19T07:15:33.484Z,2026-10-19T07:15:33.609Z,5cb7d1df-01f5-40aa-afec-e9af8932ca3c,ACC1012,7064.78,Online Shopping,Chennai,2026-10-19T07:15:33.331794Z,DEV116,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,0,468,2026-10-19T07:15:31.319Z,2026-10-19T07:15:33.609Z,c080e6ca-3554-4928-974f-2687dc8bfe0d,ACC1018,111462.22,Restaurant,Chennai,2026-10-19T07:15:31.226481Z,DEV119,true,HIGH_VALUE,2026-10-19T07:16:03.817Z
banking-transactions,1,470,2026-10-19T07:15:37.813Z,2026-10-19T07:15:42.985Z,9ac8dea1-e8de-4977-981f-10dc3758a0e6,ACC1012,7308.98,Grocery Store,Chennai,2026-10-19T07:15:37.676898Z,DEV108,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,0,474,2026-10-19T07:15:57.666Z,2026-10-19T07:15:59.527Z,dfb85b37-5698-4454-b314-31c91fb1ad07,ACC1019,2810.91,Grocery Store,Chennai,2026-10-19T07:15:57.571841Z,DEV108,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,1,473,2026-10-19T07:15:51.128Z,2026-10-19T07:15:53.455Z,3948f908-2962-4d17-812b-70755494c9ce,ACC1005,6095.86,Fuel Station,Chennai,2026-10-19T07:15:50.987548Z,DEV119,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,0,469,2026-10-19T07:15:35.644Z,2026-10-19T07:15:42.985Z,10d8e396-0bfd-4846-a1bf-9e14e924b8e8,ACC1006,4929.57,Fuel Station,Chennai,2026-10-19T07:15:35.552321Z,DEV106,false,NORMAL,2026-10-19T07:16:03.817Z
banking-transactions,1,471,2026-10-19T07:15:42.019Z,2026-10-19T07:15:42.985Z,4fa3147d-7041-452f-ab0c-65ebf0a7e15b,ACC1005,2086.65,Fuel Station,Chennai,2026-10-19T07:15:41.901768Z,DEV106,false,NORMAL,2026-10-19T07:16:03.817Z
